In [1]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

#!python dummy_data_gen.py --start-date 20260101 --end-date 20260114 --null-probability 0.05

# Petastorm Dataset Wrapper for PyTorch

Uses Petastorm's `make_batch_reader` API for efficient reading of Hive-partitioned parquet files with PyTorch integration.

**Key features**: Batch reading, filtering, shuffling, multi-worker support, and null value handling.

In [2]:
# Imports
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from petastorm import make_batch_reader
from petastorm.pytorch import DataLoader as PetastormDataLoader
import torch
import numpy as np
from pathlib import Path


In [3]:
# Petastorm uses make_batch_reader for efficient batch reading from parquet files
# Petastorm.pytorch.DataLoader provides PyTorch integration with multi-worker support
print(f"Petastorm features: batch reading, filtering, shuffling, multi-worker support")
print(f"Using make_batch_reader for efficient parquet file reading")


Petastorm features: batch reading, filtering, shuffling, multi-worker support
Using make_batch_reader for efficient parquet file reading


## Basic Usage: Read all data using explicit file listing (avoids hanging!)

In [4]:
# Create Petastorm reader using explicit file listing to avoid directory traversal hanging
from petastorm import TransformSpec
import hashlib

# Import feature config for string column handling
from feature_config import FEATURE_CONFIGS, FeatureType

data_path = Path("data").resolve()

# Use glob to explicitly list all parquet files (avoids Petastorm's slow directory discovery)
all_parquet_files = sorted(data_path.glob("**/*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files via glob")

# Convert to file:// URLs (Petastorm accepts a list of URLs)
file_urls = [f"file://{f.absolute()}" for f in all_parquet_files]

# Hash function for string bucketization (same as pyarrow_learning.ipynb)
def hash_string_to_bucket(s: str, num_buckets: int) -> int:
    """
    Hash string to bucket index 1~num_buckets. Returns 0 for null/empty.
    
    Args:
        s: String to hash (can be None or empty)
        num_buckets: Number of buckets (excluding index 0 reserved for null)
    
    Returns:
        Bucket index: 0 for null/empty, 1 to num_buckets for valid strings
    """
    if s is None or s == "":
        return 0
    hash_val = int(hashlib.md5(s.encode('utf-8')).hexdigest(), 16)
    return (hash_val % num_buckets) + 1  # 1 to num_buckets, 0 reserved for null

# Transform function to handle null values in embedding columns BEFORE Petastorm's internal batching
# This is called on each pandas DataFrame chunk before np.vstack is applied
def handle_nulls_transform(df):
    """
    Transform function to handle null values before Petastorm's internal batching.
    
    For make_batch_reader, this receives a pandas DataFrame.
    We replace None values in list columns to prevent conversion errors.
    
    IMPORTANT: Must convert ALL values to the same dtype to avoid
    'Cannot mix NumPy dtypes' error when PyArrow converts back to Arrow Table.
    
    Handles based on FeatureType from FEATURE_CONFIGS:
    - DENSE: Fill NaN with 0.0
    - EMBEDDING: Convert to float32, replace null with zero vector
    - SPARSE: Hash strings to bucket indices (int64)
    - VAR_LEN_SPARSE: Hash strings, trim/pad to max_len (int64 array)
    """
    for col, config in FEATURE_CONFIGS.items():
        if col not in df.columns:
            continue
        
        if config.type == FeatureType.DENSE:
            # Handle scalar feature columns (fill NaN with 0.0)
            df[col] = df[col].fillna(0.0)
        
        elif config.type == FeatureType.EMBEDDING:
            # Handle numeric embedding columns (list<double>)
            # Convert all values to float32 consistently
            emb_dim = config.dim
            df[col] = df[col].apply(
                lambda x, dim=emb_dim: np.asarray(x, dtype=np.float32) if x is not None else np.zeros(dim, dtype=np.float32)
            )
        
        elif config.type == FeatureType.SPARSE:
            # Handle SPARSE columns (single strings -> bucket indices)
            num_buckets = config.num_buckets
            df[col] = df[col].apply(
                lambda x, nb=num_buckets: hash_string_to_bucket(x, nb)
            )
        
        elif config.type == FeatureType.VAR_LEN_SPARSE:
            # Handle VAR_LEN_SPARSE columns (list of strings -> padded/trimmed bucket indices)
            max_len = config.max_len
            num_buckets = config.num_buckets
            
            def process_var_len_sparse(str_list, ml=max_len, nb=num_buckets):
                """Process a list of strings: hash, trim to max_len, pad with 0s."""
                if str_list is None:
                    str_list = []
                # Hash each string to bucket index
                indices = [hash_string_to_bucket(s, nb) for s in str_list]
                # Trim to max_len (keep first max_len elements)
                indices = indices[:ml]
                # Pad with 0s if shorter than max_len
                indices += [0] * (ml - len(indices))
                return np.array(indices, dtype=np.int64)
            
            df[col] = df[col].apply(process_var_len_sparse)
    
    return df

# Create TransformSpec for null handling
transform_spec = TransformSpec(func=handle_nulls_transform)

# Helper function to convert Petastorm batch (list of rows) to PyTorch tensors
def collate_fn(batch_list):
    """Convert list of Petastorm rows (namedtuples) to dict of PyTorch tensors.
    
    PetastormDataLoader passes a list of individual rows to collate_fn,
    NOT a pre-batched dictionary. Each row is a namedtuple.
    
    Handles columns based on FeatureType from FEATURE_CONFIGS:
    - VAR_LEN_SPARSE: Stack numpy arrays into 2D tensor (already processed by transform)
    - SPARSE: Convert bucket indices to int64 tensor
    - EMBEDDING: Handle object arrays robustly with NaN fallback
    - Others: Direct tensor conversion
    
    This follows the same pattern as pyarrow_dataset.py for consistency.
    """
    # Get field names from the first row (dict)
    keys = batch_list[0].keys()
    
    # Convert list of rows to dict of lists
    batch_dict = {key: [row[key] for row in batch_list] for key in keys}
    
    result = {}
    for key, values in batch_dict.items():
        # Get feature config if available
        config = FEATURE_CONFIGS.get(key)
        
        # Handle VAR_LEN_SPARSE columns - already numpy arrays from transform
        if config and config.type == FeatureType.VAR_LEN_SPARSE:
            # Values are already numpy arrays with shape (max_len,)
            # Stack them into a 2D array (batch_size, max_len)
            if len(values) > 0 and isinstance(values[0], np.ndarray):
                result[key] = torch.as_tensor(np.stack(values), dtype=torch.int64)
            else:
                # Fallback: convert to array and handle
                arr = np.array(values)
                result[key] = torch.as_tensor(arr, dtype=torch.int64)
            continue
        
        # Handle SPARSE columns - already int64 bucket indices
        if config and config.type == FeatureType.SPARSE:
            arr = np.array(values, dtype=np.int64)
            result[key] = torch.as_tensor(arr, dtype=torch.int64)
            continue
        
        # Convert list to numpy array
        arr = np.array(values)
        
        # Handle object arrays (embeddings/lists) - fallback for any remaining object arrays
        if arr.dtype == object:
            n = len(arr)
            # Determine embedding dimension from first non-null, non-empty row
            emb_dim = None
            for v in arr:
                if v is not None and hasattr(v, '__len__') and len(v) > 0:
                    emb_dim = len(v)
                    break
            
            if emb_dim is None:
                # All rows are null/empty - create empty array
                out = np.empty((n, 0), dtype=np.float32)
            else:
                # Pre-allocate with NaN, then fill non-null rows
                out = np.full((n, emb_dim), np.nan, dtype=np.float32)
                for i, v in enumerate(arr):
                    if v is not None and hasattr(v, '__len__') and len(v) > 0:
                        row_arr = np.asarray(v, dtype=np.float32)
                        ncopy = min(len(row_arr), emb_dim)
                        out[i, :ncopy] = row_arr[:ncopy]
            
            result[key] = torch.as_tensor(out)
        else:
            result[key] = torch.as_tensor(arr)
    
    return result

# Create Petastorm batch reader with TransformSpec for null handling
reader = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Pass list of file URLs instead of directory URL
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

# Create Petastorm DataLoader
loader = PetastormDataLoader(
    reader,
    batch_size=8,
    collate_fn=collate_fn
)

# Get first batch
batch = next(iter(loader))

print("\nBatch keys:", list(batch.keys()))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

print(f"\nDate in this batch: {batch['ds'][0].item()} (single batch typically comes from one file)")
print(f"\nEmbedding features:")
print(f"  emb_1: shape {batch['emb_1'].shape}, dtype={batch['emb_1'].dtype}")
print(f"  emb_2: shape {batch['emb_2'].shape}, dtype={batch['emb_2'].dtype}")

# Cleanup reader
reader.stop()



Found 1344 parquet files via glob

Batch keys: ['ds', 'h', 'swiper_id', 'swipee_id', 'feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'emb_1', 'emb_2', 'employer', 'school_name', 'interests', 'skills', 'label']

Batch shapes:
  ds: torch.Size([8]), dtype=torch.int32
  h: torch.Size([8]), dtype=torch.int32
  swiper_id: torch.Size([8]), dtype=torch.int64
  swipee_id: torch.Size([8]), dtype=torch.int64
  feat1: torch.Size([8]), dtype=torch.float64
  feat2: torch.Size([8]), dtype=torch.float64
  feat3: torch.Size([8]), dtype=torch.float64
  feat4: torch.Size([8]), dtype=torch.float64
  feat5: torch.Size([8]), dtype=torch.float64
  emb_1: torch.Size([8, 32]), dtype=torch.float32
  emb_2: torch.Size([8, 32]), dtype=torch.float32
  employer: torch.Size([8]), dtype=torch.int64
  school_name: torch.Size([8]), dtype=torch.int64
  interests: torch.Size([8, 10]), dtype=torch.int64
  skills: torch.Size([8, 10]), dtype=torch.int64
  label: torch.Size([8]), dtype=torch.int32

Date in this batch: 20260101

## Filtering: Read only specific date range

In [5]:
# Filter to only dates 20260101-20260107 using glob-based pre-filtering
# This is more efficient than letting Petastorm discover all files and then filter
dates_to_load = [f"202601{d:02d}" for d in range(1, 8)]  # 20260101 to 20260107

# Use glob to select only files for the desired dates
filtered_files = []
for ds in dates_to_load:
    filtered_files.extend(data_path.glob(f"ds={ds}/**/*.parquet"))

filtered_urls = [f"file://{f.absolute()}" for f in sorted(filtered_files)]
print(f"Found {len(filtered_urls)} files for dates {dates_to_load[0]}-{dates_to_load[-1]}")

reader_filtered = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Pass list of file URLs instead of directory URL
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_filtered = PetastormDataLoader(
    reader_filtered,
    batch_size=8,
    collate_fn=collate_fn
)

# Collect multiple batches to verify filtering works across dates
all_dates = set()
for i, batch in enumerate(loader_filtered):
    all_dates.update(batch['ds'].unique().tolist())
    if i >= 100:  # Sample first 100 batches
        break

print(f"Unique dates seen in first 100 batches: {sorted(all_dates)}")
print(f"✓ All dates are within filter range [20260101, 20260107]")

# Cleanup reader
reader_filtered.stop()

Found 672 files for dates 20260101-20260107
Unique dates seen in first 100 batches: [20260101]
✓ All dates are within filter range [20260101, 20260107]


## Shuffling: Shuffle row groups and/or rows

In [6]:
# Shuffle row groups (parquet fragments) and rows within batches
reader_shuffled = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,  
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=True,  # Shuffle order of parquet files
    shuffle_rows=True,         # Shuffle rows within each batch
    seed=42                    # For reproducibility
)

loader_shuffled = PetastormDataLoader(
    reader_shuffled,
    batch_size=8,
    collate_fn=collate_fn
)

# Get first few batches to see shuffling effect
# (fragments are shuffled, so dates won't be sequential)
print("First 5 batches - dates (should be non-sequential due to shuffling):")
for i, batch in enumerate(loader_shuffled):
    if i >= 5:
        break
    print(f"  Batch {i+1}: ds={batch['ds'][0].item()}, h={batch['h'][0].item()}")

# Cleanup reader
reader_shuffled.stop()

First 5 batches - dates (should be non-sequential due to shuffling):
  Batch 1: ds=20260111, h=0
  Batch 2: ds=20260111, h=0
  Batch 3: ds=20260111, h=0
  Batch 4: ds=20260111, h=0
  Batch 5: ds=20260111, h=0


## Null Value Detection

Show null values are handled

In [8]:
# Read data and check for null values
import torch
import numpy as np

# Create Petastorm reader using explicit file list
reader_null_check = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,
    shuffle_row_groups=False,
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_rows=False
)

loader_null_check = PetastormDataLoader(
    reader_null_check,
    batch_size=1024,
    collate_fn=collate_fn
)

# Get a batch and inspect for nulls/NaNs
print("Reading batch and checking for null values...\n")
batch = next(iter(loader_null_check))

# Check for NaN values in feature columns (Petastorm converts nulls to NaN for float columns)
feature_cols = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5']
print("Null/NaN detection in feature columns:")
print("-" * 60)
for col in feature_cols:
    if col in batch:
        tensor = batch[col]
        nan_count = torch.isnan(tensor).sum().item()
        total_count = tensor.numel()
        nan_percentage = (nan_count / total_count) * 100 if total_count > 0 else 0
        print(f"{col:10s}: {nan_count:6d} NaNs out of {total_count:6d} values ({nan_percentage:5.2f}%)")

# Check for null/NaN values in embedding columns (entire vectors can be null)
print("\nNull/NaN detection in embedding columns:")
print("-" * 60)
for emb_col in ['emb_1', 'emb_2']:
    if emb_col in batch:
        tensor = batch[emb_col]  # Shape: (batch_size, 32)
        # Count rows where entire embedding vector is NaN (all 32 dims are NaN)
        nan_rows = torch.isnan(tensor).all(dim=1).sum().item()
        total_rows = tensor.shape[0]
        nan_percentage = (nan_rows / total_rows) * 100 if total_rows > 0 else 0
        print(f"{emb_col:10s}: {nan_rows:6d} null vectors out of {total_rows:6d} rows ({nan_percentage:5.2f}%)")
        # Also show total NaN values across all dimensions
        total_nans = torch.isnan(tensor).sum().item()
        total_values = tensor.numel()
        print(f"           {total_nans:6d} NaN values out of {total_values:6d} total ({total_nans/total_values*100:5.2f}%)")

print(f"\nTotal batch size: {len(batch['ds'])} rows")
print(f"\nSample values from feat1 (showing first 50):")
print(batch['feat1'][:50])
nan_indices = torch.isnan(batch['feat1']).nonzero(as_tuple=True)[0]
if len(nan_indices) > 0:
    print(f"\nNaN positions in feat1 (first 20): {nan_indices[:20].tolist()}")
else:
    print(f"\nNo NaN values found in feat1 for this batch")

# Cleanup reader
reader_null_check.stop()

Reading batch and checking for null values...

Null/NaN detection in feature columns:
------------------------------------------------------------
feat1     :      0 NaNs out of   1024 values ( 0.00%)
feat2     :      0 NaNs out of   1024 values ( 0.00%)
feat3     :      0 NaNs out of   1024 values ( 0.00%)
feat4     :      0 NaNs out of   1024 values ( 0.00%)
feat5     :      0 NaNs out of   1024 values ( 0.00%)

Null/NaN detection in embedding columns:
------------------------------------------------------------
emb_1     :      0 null vectors out of   1024 rows ( 0.00%)
                0 NaN values out of  32768 total ( 0.00%)
emb_2     :      0 null vectors out of   1024 rows ( 0.00%)
                0 NaN values out of  32768 total ( 0.00%)

Total batch size: 1024 rows

Sample values from feat1 (showing first 50):
tensor([ 0.0000, -0.4528, -0.3938, -0.4573,  0.8543, -0.1159,  0.3312,  2.1075,
         1.3726, -0.7110,  0.3121,  1.3129, -1.1738,  1.0973,  0.3052,  0.8699,
         

## Working with Embedding Features

Demonstrate how to use the embedding features (emb_1, emb_2) in your models.

In [10]:
# Example: Working with embedding features
reader_emb = make_batch_reader(
    dataset_url_or_urls=file_urls,  # Use explicit file list
    num_epochs=1,
    workers_count=4,
    transform_spec=transform_spec,  # Apply null handling transformation
    shuffle_row_groups=False,
    shuffle_rows=False
)

loader_emb = PetastormDataLoader(
    reader_emb,
    batch_size=32,
    collate_fn=collate_fn
)

batch = next(iter(loader_emb))

print("Embedding feature shapes:")
print(f"  emb_1: {batch['emb_1'].shape} (batch_size={batch['emb_1'].shape[0]}, emb_dim={batch['emb_1'].shape[1]})")
print(f"  emb_2: {batch['emb_2'].shape} (batch_size={batch['emb_2'].shape[0]}, emb_dim={batch['emb_2'].shape[1]})")

print(f"\nExample usage:")
print(f"  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape {torch.cat([batch['emb_1'], batch['emb_2']], dim=1).shape}")
print(f"  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape {(batch['emb_1'] + batch['emb_2']).shape}")
print(f"  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape {torch.sum(batch['emb_1'] * batch['emb_2'], dim=1).shape}")

# Show sample embedding values (first row, first 5 dimensions)
print(f"\nSample embedding values (first row):")
print(f"  emb_1[:5]: {batch['emb_1'][0, :5].tolist()}")
print(f"  emb_2[:5]: {batch['emb_2'][0, :5].tolist()}")

# Check for any null vectors
null_emb_1 = torch.isnan(batch['emb_1']).all(dim=1).sum().item()
null_emb_2 = torch.isnan(batch['emb_2']).all(dim=1).sum().item()
print(f"\nNull vectors in this batch: emb_1={null_emb_1}, emb_2={null_emb_2}")

# Cleanup reader
reader_emb.stop()

Embedding feature shapes:
  emb_1: torch.Size([32, 32]) (batch_size=32, emb_dim=32)
  emb_2: torch.Size([32, 32]) (batch_size=32, emb_dim=32)

Example usage:
  - Concatenate embeddings: torch.cat([batch['emb_1'], batch['emb_2']], dim=1) -> shape torch.Size([32, 64])
  - Element-wise operations: batch['emb_1'] + batch['emb_2'] -> shape torch.Size([32, 32])
  - Dot product: torch.sum(batch['emb_1'] * batch['emb_2'], dim=1) -> shape torch.Size([32])

Sample embedding values (first row):
  emb_1[:5]: [0.5495064854621887, -1.2492564916610718, -0.9102044105529785, -0.1586080938577652, 0.32569417357444763]
  emb_2[:5]: [-0.12696701288223267, -0.5865147113800049, 0.4995621144771576, 0.26829472184181213, -0.9789350628852844]

Null vectors in this batch: emb_1=0, emb_2=0
